<a href="https://www.kaggle.com/code/samithsachidanandan/dropout-vs-without-dropout?scriptVersionId=311877610" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# 1. Generate dataset
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Standardize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 3. Convert to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1)

# 4. Define model with Dropout
class DropoutNet(nn.Module):
    def __init__(self, dropout_p=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),   # <-- dropout here
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),   # <-- and here
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model = DropoutNet(dropout_p=0.5)

# 5. Train
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    output = model(X_train_t)
    loss = criterion(output, y_train_t)
    loss.backward()
    optimizer.step()

# 6. Evaluate
model.eval()   # disables dropout at test time
with torch.no_grad():
    preds = (model(X_test_t) > 0.5).float()
    acc = (preds == y_test_t).float().mean()
    print(f"Test Accuracy: {acc.item():.4f}")

Test Accuracy: 0.8500


In [2]:
# Train the same architecture without dropout for comparison
class PlainNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1),  nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

plain_model = PlainNet()
optimizer2 = optim.Adam(plain_model.parameters(), lr=0.001)

for epoch in range(100):
    plain_model.train()
    optimizer2.zero_grad()
    loss = criterion(plain_model(X_train_t), y_train_t)
    loss.backward()
    optimizer2.step()

plain_model.eval()
with torch.no_grad():
    preds2 = (plain_model(X_test_t) > 0.5).float()
    acc2 = (preds2 == y_test_t).float().mean()
    print(f"Test Accuracy (no dropout): {acc2.item():.4f}")

Test Accuracy (no dropout): 0.8600
